In [38]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

DATA_DIR = Path("../data")

transactions = pd.read_parquet(
    DATA_DIR / "online_retail_II_cleaned.parquet"
)

print(transactions.shape)
display(transactions.head())
display(transactions.dtypes)

(779425, 11)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancelled_invoice,line_value
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,Year 2009-2010,False,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010,False,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010,False,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,Year 2009-2010,False,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,Year 2009-2010,False,30.0


Invoice                 string[python]
StockCode               string[python]
Description             string[python]
Quantity                         int64
InvoiceDate             datetime64[ns]
Price                          float64
Customer ID             string[python]
Country                 string[python]
source_sheet            string[python]
is_cancelled_invoice              bool
line_value                     float64
dtype: object

In [39]:
assert transactions["Customer ID"].notna().all()
assert transactions["Quantity"].gt(0).all()
assert transactions["Price"].gt(0).all()
assert not transactions["is_cancelled_invoice"].any()
assert not transactions.duplicated().any()

In [40]:
transactions["order_id"] = (
    transactions["Invoice"]
    .astype("string")
    .str.strip()
)

In [41]:
order_integrity = (
    transactions
    .groupby("order_id")
    .agg(
        customers=("Customer ID", "nunique"),
        timestamps=("InvoiceDate", "nunique"),
        source_sheets=("source_sheet", "nunique"),
    )
)

display(order_integrity["customers"].value_counts())
display(order_integrity["timestamps"].value_counts())
display(order_integrity["source_sheets"].value_counts())

assert order_integrity["customers"].max() == 1

multi_timestamp_invoices = (
    order_integrity["timestamps"] > 1
).sum()

print(
    "Invoices containing multiple timestamps:",
    multi_timestamp_invoices
)

customers
1    36969
Name: count, dtype: int64

timestamps
1    36905
2       64
Name: count, dtype: int64

source_sheets
1    36969
Name: count, dtype: int64

Invoices containing multiple timestamps: 64


In [42]:
order_timestamp_span = (
    transactions
    .groupby("order_id")["InvoiceDate"]
    .agg(["min", "max"])
)

order_timestamp_span["span_minutes"] = (
    order_timestamp_span["max"]
    - order_timestamp_span["min"]
).dt.total_seconds() / 60

print(
    "Maximum within-invoice timestamp span:",
    order_timestamp_span["span_minutes"].max(),
    "minutes"
)

assert order_timestamp_span["span_minutes"].max() <= 2

Maximum within-invoice timestamp span: 2.0 minutes


### Invoices with multiple timestamps

Sixty-four invoice numbers contained product lines recorded across two
timestamps. All belonged to a single customer, and the maximum difference
within an invoice was two minutes. These records were therefore treated as
single orders, using the earliest timestamp as the order time and aggregating
all product lines under the invoice number.

In [43]:
customers_per_order = (
    transactions.groupby("order_id")["Customer ID"].nunique()
)

customers_per_order.value_counts()

Customer ID
1    36969
Name: count, dtype: int64

In [44]:
assert customers_per_order.max() == 1

In [45]:
dates_per_order = (
    transactions.groupby("order_id")["InvoiceDate"].nunique()
)

dates_per_order.value_counts()

InvoiceDate
1    36905
2       64
Name: count, dtype: int64

In [46]:
orders = (
    transactions
    .groupby("order_id", as_index=False)
    .agg(
        customer_id=("Customer ID", "first"),
        order_date=("InvoiceDate", "min"),
        order_value=("line_value", "sum"),
        total_quantity=("Quantity", "sum"),
        unique_products=("StockCode", "nunique"),
        product_lines=("StockCode", "size"),
        country=("Country", "first"),
    )
)

In [47]:
orders["average_item_price"] = (
    orders["order_value"] / orders["total_quantity"]
)

In [48]:
print(f"Transaction rows: {len(transactions):,}")
print(f"Valid orders: {len(orders):,}")
print(f"Customers: {orders['customer_id'].nunique():,}")

display(orders.head())
display(orders.describe())

Transaction rows: 779,425
Valid orders: 36,969
Customers: 5,878


,order_id,customer_id,order_date,order_value,total_quantity,unique_products,product_lines,country,average_item_price
0,489434,13085,2009-12-01 07:45:00,505.30,166,8,8,United Kingdom,3.043976
1,489435,13085,2009-12-01 07:46:00,145.80,60,4,4,United Kingdom,2.430000
2,489436,13078,2009-12-01 09:06:00,630.33,193,19,19,United Kingdom,3.265959
3,489437,15362,2009-12-01 09:08:00,310.75,145,23,23,United Kingdom,2.143103
4,489438,18102,2009-12-01 09:24:00,2286.24,826,17,17,United Kingdom,2.767845


,order_date,order_value,total_quantity,unique_products,product_lines,average_item_price
count,36969,36969.000000,36969.000000,36969.000000,36969.000000,36969.000000
mean,2010-12-26 21:07:25.743731456,469.983074,284.399145,20.797560,21.083205,6.859504
min,2009-12-01 07:45:00,0.380000,1.000000,1.000000,1.000000,0.060000
25%,2010-06-29 11:56:00,157.920000,72.000000,6.000000,6.000000,1.429184
50%,2010-12-01 12:08:00,303.040000,151.000000,15.000000,15.000000,1.927358
75%,2011-07-13 11:20:00,477.280000,287.000000,27.000000,27.000000,2.691186
max,2011-12-09 12:50:00,168469.600000,87167.000000,541.000000,542.000000,10953.500000
std,NaN,1359.741927,1232.382259,22.399873,22.964763,131.701566


In [49]:
assert orders["order_id"].is_unique
assert orders["customer_id"].notna().all()
assert orders["order_value"].gt(0).all()
assert orders["total_quantity"].gt(0).all()
assert orders["unique_products"].ge(1).all()

In [50]:
orders.nlargest(
    20, "order_value"
)[
    [
        "order_id",
        "customer_id",
        "order_date",
        "order_value",
        "total_quantity",
        "unique_products",
        "country",
    ]
]

,order_id,customer_id,order_date,order_value,total_quantity,unique_products,country
36936,581483,16446,2011-12-09 09:15:00,168469.60,80995,1,United Kingdom
20346,541431,12346,2011-01-18 10:01:00,77183.60,74215,1,United Kingdom
1604,493819,14156,2010-01-07 12:34:00,44051.60,25018,94,EIRE
26362,556444,15098,2011-06-10 15:28:00,38970.00,60,1,United Kingdom
13428,524181,17450,2010-09-27 16:59:00,33167.80,8172,13,United Kingdom
30854,567423,17450,2011-09-20 11:05:00,31698.16,12572,12,United Kingdom
14625,526934,18102,2010-10-14 09:46:00,26007.08,5079,15,United Kingdom
10023,515944,18102,2010-07-15 15:29:00,22863.36,4992,17,United Kingdom
26548,556917,12415,2011-06-15 13:37:00,22775.93,15049,138,Australia
32893,572209,18102,2011-10-21 12:08:00,22206.00,1920,7,United Kingdom


In [51]:
orders = orders.sort_values(
    ["customer_id", "order_date", "order_id"]
).reset_index(drop=True)

orders["order_number"] = (
    orders.groupby("customer_id").cumcount() + 1
)

In [52]:
repeat_customer_ids = (
    orders.groupby("customer_id")
          .size()
          .loc[lambda x: x >= 2]
          .index[:5]
)

orders[
    orders["customer_id"].isin(repeat_customer_ids)
].head(20)

,order_id,customer_id,order_date,order_value,total_quantity,unique_products,product_lines,country,average_item_price,order_number
0,491725,12346,2009-12-14 08:34:00,45.00,10,1,1,United Kingdom,4.500000,1
1,491742,12346,2009-12-14 11:00:00,22.50,5,1,1,United Kingdom,4.500000,2
2,491744,12346,2009-12-14 11:02:00,22.50,5,1,1,United Kingdom,4.500000,3
3,492718,12346,2009-12-18 10:47:00,22.50,5,1,1,United Kingdom,4.500000,4
4,492722,12346,2009-12-18 10:55:00,1.00,1,1,1,United Kingdom,1.000000,5
5,493410,12346,2010-01-04 09:24:00,22.50,5,1,1,United Kingdom,4.500000,6
6,493412,12346,2010-01-04 09:53:00,22.50,5,1,1,United Kingdom,4.500000,7
7,494450,12346,2010-01-14 13:50:00,22.50,5,1,1,United Kingdom,4.500000,8
8,495295,12346,2010-01-22 13:30:00,22.50,5,1,1,United Kingdom,4.500000,9
9,499763,12346,2010-03-02 13:08:00,27.05,5,5,5,United Kingdom,5.410000,10


In [53]:
first_orders = (
    orders.loc[orders["order_number"] == 1]
    .copy()
    .rename(columns={
        "order_id": "first_order_id",
        "order_date": "first_order_date",
        "order_value": "first_order_value",
        "total_quantity": "first_order_quantity",
        "unique_products": "first_order_unique_products",
        "product_lines": "first_order_product_lines",
        "average_item_price": "first_order_average_item_price",
        "country": "first_order_country",
    })
)

In [54]:
first_orders = first_orders[
    [
        "customer_id",
        "first_order_id",
        "first_order_date",
        "first_order_value",
        "first_order_quantity",
        "first_order_unique_products",
        "first_order_product_lines",
        "first_order_average_item_price",
        "first_order_country",
    ]
]

In [55]:
first_orders["first_order_weekday"] = (
    first_orders["first_order_date"].dt.day_name()
)

first_orders["first_order_month"] = (
    first_orders["first_order_date"].dt.month
)

first_orders["first_order_hour"] = (
    first_orders["first_order_date"].dt.hour
)

first_orders["first_order_is_weekend"] = (
    first_orders["first_order_date"].dt.dayofweek >= 5
)

In [56]:
later_order_candidates = orders.merge(
    first_orders[
        ["customer_id", "first_order_date"]
    ],
    on="customer_id",
    how="left",
    validate="many_to_one",
)

later_order_candidates = later_order_candidates[
    later_order_candidates["order_date"]
    > later_order_candidates["first_order_date"]
]

second_orders = (
    later_order_candidates
    .groupby("customer_id", as_index=False)
    .agg(
        second_order_date=("order_date", "min")
    )
)

In [57]:
customers = first_orders.merge(
    second_orders,
    on="customer_id",
    how="left",
    validate="one_to_one",
)

In [58]:
customers["days_to_second_order"] = (
    customers["second_order_date"]
    - customers["first_order_date"]
).dt.total_seconds() / (24 * 60 * 60)

In [59]:
assert customers[
    "days_to_second_order"
].dropna().gt(0).all()

In [60]:
customers[
    [
        "customer_id",
        "first_order_date",
        "second_order_date",
        "days_to_second_order",
    ]
].head(20)

,customer_id,first_order_date,second_order_date,days_to_second_order
0,12346,2009-12-14 08:34:00,2009-12-14 11:00:00,0.101389
1,12347,2010-10-31 14:20:00,2010-12-07 14:57:00,37.025694
2,12348,2010-09-27 14:59:00,2010-12-16 19:09:00,80.173611
3,12349,2010-04-29 13:20:00,2010-05-18 09:57:00,18.859028
4,12350,2011-02-02 16:01:00,NaT,NaN
5,12351,2010-11-29 15:23:00,NaT,NaN
6,12352,2010-11-12 10:20:00,2010-11-29 10:07:00,16.990972
7,12353,2010-10-27 12:44:00,2011-05-19 17:47:00,204.210417
8,12354,2011-04-21 13:11:00,NaT,NaN
9,12355,2010-05-21 11:59:00,2011-05-09 13:49:00,353.076389


In [61]:
data_end_date = transactions["InvoiceDate"].max()

data_end_date

Timestamp('2011-12-09 12:50:00')

In [62]:
customers["available_followup_days"] = (
    data_end_date - customers["first_order_date"]
).dt.total_seconds() / (24 * 60 * 60)

In [63]:
customers["has_full_90d_followup"] = (
    customers["available_followup_days"] >= 90
)

In [64]:
customers["has_full_90d_followup"].value_counts()

has_full_90d_followup
True     5281
False     597
Name: count, dtype: int64

In [65]:
eligible_customers = (
    customers.loc[customers["has_full_90d_followup"]]
    .copy()
)

In [66]:
eligible_customers["repeat_within_90d"] = (
    eligible_customers["days_to_second_order"]
    .between(0, 90, inclusive="right")
    .astype(int)
)

1 = second valid order occurred within 90 days
0 = no second valid order occurred within 90 days

In [67]:
eligible_customers["repeat_within_90d"].value_counts()
eligible_customers["repeat_within_90d"].value_counts(normalize=True)

repeat_within_90d
0    0.529635
1    0.470365
Name: proportion, dtype: float64

In [68]:
eligible_customers.loc[
    eligible_customers["repeat_within_90d"] == 1,
    [
        "customer_id",
        "first_order_date",
        "second_order_date",
        "days_to_second_order",
    ]
].head()

,customer_id,first_order_date,second_order_date,days_to_second_order
0,12346,2009-12-14 08:34:00,2009-12-14 11:00:00,0.101389
1,12347,2010-10-31 14:20:00,2010-12-07 14:57:00,37.025694
2,12348,2010-09-27 14:59:00,2010-12-16 19:09:00,80.173611
3,12349,2010-04-29 13:20:00,2010-05-18 09:57:00,18.859028
6,12352,2010-11-12 10:20:00,2010-11-29 10:07:00,16.990972


In [69]:
eligible_customers.loc[
    eligible_customers["repeat_within_90d"] == 0,
    [
        "customer_id",
        "first_order_date",
        "second_order_date",
        "days_to_second_order",
    ]
].head()

,customer_id,first_order_date,second_order_date,days_to_second_order
4,12350,2011-02-02 16:01:00,NaT,NaN
5,12351,2010-11-29 15:23:00,NaT,NaN
7,12353,2010-10-27 12:44:00,2011-05-19 17:47:00,204.210417
8,12354,2011-04-21 13:11:00,NaT,NaN
9,12355,2010-05-21 11:59:00,2011-05-09 13:49:00,353.076389


In [70]:
assert eligible_customers["customer_id"].is_unique
assert eligible_customers["available_followup_days"].ge(90).all()
assert eligible_customers["repeat_within_90d"].isin([0, 1]).all()

assert (
    eligible_customers.loc[
        eligible_customers["repeat_within_90d"] == 1,
        "days_to_second_order"
    ].le(90).all()
)

In [71]:
feature_columns = [
    "first_order_value",
    "first_order_quantity",
    "first_order_unique_products",
    "first_order_product_lines",
    "first_order_average_item_price",
    "first_order_country",
    "first_order_weekday",
    "first_order_month",
    "first_order_hour",
    "first_order_is_weekend",
]

eligible_customers[feature_columns].head()

,first_order_value,first_order_quantity,first_order_unique_products,first_order_product_lines,first_order_average_item_price,first_order_country,first_order_weekday,first_order_month,first_order_hour,first_order_is_weekend
0,45.00,10,1,1,4.500000,United Kingdom,Monday,12,8,False
1,611.53,509,40,40,1.201434,Iceland,Sunday,10,14,True
2,222.16,373,20,20,0.595603,Finland,Monday,9,14,False
3,1068.52,473,46,46,2.259027,Italy,Thursday,4,13,False
4,334.40,197,17,17,1.697462,Norway,Wednesday,2,16,False


In [72]:
output_path = DATA_DIR / "customer_retention_dataset.parquet"

eligible_customers.to_parquet(
    output_path,
    index=False,
)

check = pd.read_parquet(output_path)

assert check.shape == eligible_customers.shape

print(f"Saved {len(check):,} eligible customers")

Saved 5,281 eligible customers


In [73]:
print("Orders:", len(orders))
print("Customers before censoring:", len(customers))
print("Eligible customers:", len(eligible_customers))
print(
    "90-day repeat rate:",
    eligible_customers["repeat_within_90d"].mean()
)
print(
    "Exact zero-day second orders:",
    customers["days_to_second_order"].eq(0).sum()
)

print(
    "Second orders within 24 hours:",
    customers["days_to_second_order"]
    .between(0, 1, inclusive="both")
    .sum()
)

Orders: 36969
Customers before censoring: 5878
Eligible customers: 5281
90-day repeat rate: 0.47036546108691535
Exact zero-day second orders: 0
Second orders within 24 hours: 331


## Explainations  
Valid orders: 37745. Distinct non-cancelled, positive-value invoices.   
Customers: 5878. Identifiable customers with at least one valid order.  
Eligible customers: 5281. Customers whose first order has at least 90 days of follow-up.  
Excluded by censoring: 597. Insufficient time to observe the full outcome.  
90-day repeat rate: 46.87%. Eligible customers with a second order within 90 days.  